In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]
os.chdir(PROJECT_ROOT)

In [3]:
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

from src.schemas import RAGResponse, SourceCitation
from src.vectorstore import format_docs, get_retriever, load_vectorstore

In [4]:
load_dotenv()

True

In [5]:
DEFAULT_PROMPT_PATH = Path("prompts/rag_prompt.txt")
DEFAULT_CHAT_MODEL = "gpt-4o-mini"

In [6]:
# load rag prompt
prompt_path = "prompts/rag_prompt.txt"
path = Path(prompt_path)

if not path.exists():
    raise FileNotFoundError(f"RAG prompt file not found: {path}")

template = path.read_text(encoding="utf-8")
template

'You are a helpful assistant that answers questions using the provided document context.\n\nRules:\n1. Answer using ONLY the information in the Document Context below.\n2. Do NOT invent facts that are not supported by the Document Context.\n3. If the Document Context does not contain enough information to answer, say:\n   "I could not find that information in the provided documents."\n4. Use the Conversation History when it helps you understand follow-up questions\n   (for example, what "it" or "they" refers to).\n5. Keep your answer clear and concise.\n\nConversation History:\n{chat_history}\n\nDocument Context:\n{context}\n\nQuestion:\n{question}\n\nAnswer:\n'

In [7]:
prompt = PromptTemplate(
    template=template,
    input_variables=["context", "chat_history", "question"]
)

In [8]:
prompt

PromptTemplate(input_variables=['chat_history', 'context', 'question'], input_types={}, partial_variables={}, template='You are a helpful assistant that answers questions using the provided document context.\n\nRules:\n1. Answer using ONLY the information in the Document Context below.\n2. Do NOT invent facts that are not supported by the Document Context.\n3. If the Document Context does not contain enough information to answer, say:\n   "I could not find that information in the provided documents."\n4. Use the Conversation History when it helps you understand follow-up questions\n   (for example, what "it" or "they" refers to).\n5. Keep your answer clear and concise.\n\nConversation History:\n{chat_history}\n\nDocument Context:\n{context}\n\nQuestion:\n{question}\n\nAnswer:\n')

In [9]:
from pydantic import SecretStr

In [10]:
# Calling model

api_key = os.getenv("OPENAI_API_KEY", "").strip()
if not api_key:
    raise ValueError(
        "OPENAI_API_KEY is missing. Add it to your .env file before chatting."
    )
model_name = os.getenv("OPENAI_CHAT_MODEL", DEFAULT_CHAT_MODEL)

llm = ChatOpenAI(
            model=model_name,
            api_key=SecretStr(api_key),
            temperature=0,
        )

In [11]:
question = "Please tell me about basics of the leave policy"

cleaned_question = (question or "").strip()

history: list[dict[str, str]] = []

vector_store = load_vectorstore()

retriever = get_retriever(vectorstore=vector_store, k=2)

retrieved_docs = retriever.invoke(cleaned_question)

In [12]:
retrieved_docs

[Document(id='9fc45271-1cb5-4d95-a50e-6d9b11af5789', metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20260905123243', 'source': 'company_handbook.pdf', 'total_pages': 2, 'page': 1, 'page_label': '1', 'filename': 'company_handbook.pdf'}, page_content='Aspire Demo Company Handbook\nWelcome to Aspire Demo Company. This handbook explains working hours, leave policy, remote\nwork, and employee benefits for all full-time staff.\n1. Working Hours\nStandard working hours are Monday to Friday, 9:00 AM to 6:00 PM, including a one-hour lunch\nbreak. Core collaboration hours are 10:00 AM to 4:00 PM. Employees may request flexible start\ntimes between 8:00 AM and 10:00 AM with manager approval.\n2. Leave Policy\nFull-time employees receive 20 days of paid annual leave each calendar year. In addition,\nemployees receive 10 days of paid sick leave. Leave requests should be submitted at least 7 days\nin advance for planned vacations. Unused an

In [13]:
context = format_docs(retrieved_docs)
context

'[Source 1: company_handbook.pdf (page 1)]\nAspire Demo Company Handbook\nWelcome to Aspire Demo Company. This handbook explains working hours, leave policy, remote\nwork, and employee benefits for all full-time staff.\n1. Working Hours\nStandard working hours are Monday to Friday, 9:00 AM to 6:00 PM, including a one-hour lunch\nbreak. Core collaboration hours are 10:00 AM to 4:00 PM. Employees may request flexible start\ntimes between 8:00 AM and 10:00 AM with manager approval.\n2. Leave Policy\nFull-time employees receive 20 days of paid annual leave each calendar year. In addition,\nemployees receive 10 days of paid sick leave. Leave requests should be submitted at least 7 days\nin advance for planned vacations. Unused annual leave may carry over a maximum of 5 days into\nthe next year.\nPage 1\n\n[Source 2: company_handbook.pdf (page 2)]\n3. Remote Work\nEmployees may work remotely up to 3 days per week after completing their first 90 days with the\ncompany. Remote work days must b

In [14]:
history = [
    {"role": "user", "content": "Hello, I am Aziz, how can you help me?"},
    {"role": "assistant", "content": "Hi Aziz, I can help answer questions and assist with tasks."},
]

In [15]:
if not history:
    print("No previous conversation.")
    lines = []
else:
    lines = []
    for msg in history:
        role = msg.get("role", "user")
        content = msg.get("content", "").strip()
        if not content:
            continue
        label = "user" if role == "user" else "assistant"
        lines.append(f"{label}: {content}")

history_text = "\n".join(lines) if lines else "No previous conversation."

In [16]:
history_text

'user: Hello, I am Aziz, how can you help me?\nassistant: Hi Aziz, I can help answer questions and assist with tasks.'

In [17]:
final_prompt = prompt.format(
    context=context,
    chat_history=history_text,
    question=cleaned_question,
)

In [18]:
result = llm.invoke(final_prompt)
answer = result.content if hasattr(result, "content") else str(result)
answer

'Full-time employees receive 20 days of paid annual leave each calendar year and 10 days of paid sick leave. Leave requests should be submitted at least 7 days in advance for planned vacations. Unused annual leave may carry over a maximum of 5 days into the next year.'

In [19]:
citations: list[SourceCitation] = []
seen: set[tuple[str, int | None]] = set()

for doc in retrieved_docs:
    filename = doc.metadata.get("filename") or doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page")
    key = (str(filename), page)
    if key in seen:
        continue
    seen.add(key)

    preview = doc.page_content.strip().replace("\n", " ")
    if len(preview) > 160:
        preview = preview[:157] + "..."

    citations.append(
        SourceCitation(
            filename=str(filename),
            page=page,
            preview=preview,
        )
    )

In [20]:
citations

[SourceCitation(filename='company_handbook.pdf', page=1, preview='Aspire Demo Company Handbook Welcome to Aspire Demo Company. This handbook explains working hours, leave policy, remote work, and employee benefits for all f...'),
 SourceCitation(filename='company_handbook.pdf', page=2, preview='3. Remote Work Employees may work remotely up to 3 days per week after completing their first 90 days with the company. Remote work days must be agreed with ...')]

In [21]:
# Saving the response in form of pydantic schema based output
rag_response = RAGResponse(
    answer=str(answer).strip(),
    sources=citations,
    num_chunks=len(retrieved_docs)
)

In [22]:
rag_response

RAGResponse(answer='Full-time employees receive 20 days of paid annual leave each calendar year and 10 days of paid sick leave. Leave requests should be submitted at least 7 days in advance for planned vacations. Unused annual leave may carry over a maximum of 5 days into the next year.', sources=[SourceCitation(filename='company_handbook.pdf', page=1, preview='Aspire Demo Company Handbook Welcome to Aspire Demo Company. This handbook explains working hours, leave policy, remote work, and employee benefits for all f...'), SourceCitation(filename='company_handbook.pdf', page=2, preview='3. Remote Work Employees may work remotely up to 3 days per week after completing their first 90 days with the company. Remote work days must be agreed with ...')], num_chunks=2)